In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier


### 1. Load the Iris dataset

In [4]:
data = load_iris()
X = data.data
y = data.target

### 2. Split into training and testing sets

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 3. Initialize the LGBMClassifier
- LGBM handles numerical data natively
- For categorical data, we would need to convert columns to 'category' dtype in a DataFrame

In [ ]:
model = LGBMClassifier(
    boosting_type='gbdt',
    n_estimators=100,
    num_leaves=30,
    min_child_samples=10, 
    learning_rate=0.05, 
    feature_fraction=0.8,
    bagging_fraction = 0.8,
    bagging_freq=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=-1 # hidden any warnings
)

### All Hyperparameters for LGBMClassifier: (See the end with more details)
- boosting_type: str = "gbdt", # default algorithm used in Gradient Boosting. iteratively adding new decision tress to themodel to correct the errors (residuals) made by the previous trees.
- num_leaves: int = 31, # maximum leaves
- max_depth: int = -1, # layers per tree
- learning_rate: float = 0.1, # scales the contribution of each tree
- n_estimators: int = 100,  # total trees
- subsample_for_bin: int = 200000, 
- objective: _LGBM_ScikitCustomObjectiveFunction | str | None = None,
- class_weight: Dict | str | None = None, 
- min_split_gain: float = 0, 
- min_child_weight: float = 0.001, 
- min_child_samples: int = 20, 
- subsample: float = 1, 
- subsample_freq: int = 0, 
- colsample_bytree: float = 1, 
- reg_alpha: float = 0, 
- reg_lambda: float = 0, 
- random_state: int | RandomState | Generator | None = None, 
- n_jobs: int | None = None, 
- importance_type: str = "split"

### 4. Train the model

In [11]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,30
,max_depth,-1
,learning_rate,0.05
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### 5. Make predictions

In [12]:
y_pred = model.predict(X_test)

### 6. Evaluate

In [13]:
accuracy = accuracy_score(y_test, y_pred)

In [15]:
for name, importance in zip(data.feature_names, model.feature_importances_):
    print(f"{name}: {importance}")

sepal length (cm): 154
sepal width (cm): 268
petal length (cm): 471
petal width (cm): 282


## Hyperparameter Details

**`model = LGBMClassifier(`**
Initializes the LightGBM Classifier object using the Scikit-learn compatible API.

**`boosting_type='gbdt',`**
Specifies the algorithm used to grow trees. `'gbdt'` stands for **Gradient Boosting Decision Tree**, which is the standard method where each new tree attempts to correct the errors (residuals) made by the previous trees.

Each new tree is trained to predict the **errors** (residuals) made by the previous trees.

**`num_leaves=30,`**
Sets the maximum number of leaves in a single tree. This is the main parameter for controlling the complexity of the model. A higher number allows the model to learn more complex patterns but increases the risk of overfitting.

**`max_depth=-1,`**
layers per tree

**`learning_rate=0.05,`**
Also known as "shrinkage." It scales the contribution of each tree. A smaller value (like 0.05) makes the learning process more conservative and stable, requiring more `n_estimators` to reach the optimal solution.

**`n_estimators=100,`**
The total number of boosting iterations (the number of trees to build). The model will add 100 trees sequentially to the ensemble.

**`subsample_for_bin`**
By default, LightGBM looks at all rows in your dataset to determine where the optimal "bin boundaries" should be. This can be computationally expensive if your dataset is massive.
- LightGBM takes a random sample of your data (in this case, 50%).
- It calculates the bin boundaries based only on that sample.
- It then uses those boundaries to bin the entire dataset for the actual training process.

**`objective`**

| Objective | Task Type | Target Variable ($y$) | Example |
| :--- | :--- | :--- | :--- |
| **`binary`** | **Binary Classification** | Two classes (0 or 1) | Is this email spam? |
| **`multiclass`** | **Multi-class Classification** | Three or more classes (0, 1, 2...) | Is this image a cat, dog, or bird? |
| **`regression`** | **Regression** | Continuous numbers | What will the house price be? |
| **`lambdarank`** | **Ranking** | Ordered list of items | Which product should appear first in search results? |

**`class_weight`**

handle the imbalanced datasets:
- None (Default): All classes are treated equally
- 'balanced': Automatically calculates weights inversely proportional to class frequencies. This forces the model to pay more attention to the minority class.
- 'Dict' (e.g. {0:1, 1:10}): manually assign a weight to each class. this means a misclassifying a 1 is 10 times more costly than misclassifying a 0.

**`min_split_gain`**
- Every time the model considers splitting a node, it calculates the "gain" (the improvement in the loss function). If the gain is less than min_split_gain, the split is discarded.
- Higher values helps prevent overfitting
- Increase this if the model is growing too many branches that caputre noise rather than real patterns.

**`min_child_weight`**
- it controls the minimum amount of 'evidence' (weight) required to create a new leaf.
- In decision trees, every leaf must represent a certain amount of "weight." In classification, this is closely related to the number of samples in that leaf.
- Higher values: Requires a leaf to have a higher "sum of weights" to be created. This prevents the model from creating leaves that represent only one or two specific data points (outliers).
- When to use: This is one of the most powerful tools to combat overfitting. If your model is creating very deep, complex trees that only apply to a few specific rows, increase min_child_weight.

These parameters are the primary tools for controlling **overfitting** and **training speed** in LightGBM. They can be categorized into three groups: **Sample/Feature Subsampling**, **Tree Growth Constraints**, and **Regularization**.

**`Subsampling (Stochastic Gradient Boosting)`**

These parameters introduce randomness into the training process to prevent the model from memorizing the training data.
*   **`subsample`** (Fraction of data)
    *   **What it is:** The fraction of the total dataset to be used for growing each tree.
    *   **Value:** `(0.0, 1.0]`. Default is `1.0` (uses all data).
    *   **Effect:** If set to `0.8`, each tree is trained on a random 80% of the data. This prevents the model from becoming too reliant on specific rows, reducing overfitting.
*   **`subsample_freq`** (Frequency of subsampling)
    *   **What it is:** Defines how often to perform subsampling.
    *   **Value:** `0` (default) or an integer.
    *   **Effect:** If `0`, no subsampling is performed. If `1`, it performs subsampling for every iteration (tree). If `5`, it performs subsampling every 5 trees. This adds more randomness to the ensemble.
*   **`colsample_bytree`** (Fraction of features)
    *   **What it is:** The fraction of features (columns) to be randomly selected for each tree.
    *   **Value:** `(0.0, 1.0]`. Default is `1.0`.
    *   **Effect:** If set to `0.7`, each tree only sees 70% of the available features. This prevents a single highly dominant feature from being used in every single tree, forcing the model to find patterns in other features.

**`Tree Growth Constraints`**

These parameters limit the complexity of the individual trees.
*   **`min_child_samples`** (Minimum data per leaf)
    *   **What it is:** The minimum number of data points (samples) required in a leaf.
    *   **Effect:** This is a very important parameter for preventing overfitting. If a split results in a leaf with fewer than `min_child_samples`, that split is discarded. 
    *   **When to use:** Increase this value if your model is creating very small, specific leaves that only apply to a handful of outliers.

**`Regularization (Weight Penalties)`**

These parameters add a penalty to the loss function based on the magnitude of the leaf weights.
*   **`reg_alpha`** (L1 Regularization)
    *   **What it is:** Adds a penalty proportional to the **absolute value** of the weights.
    *   **Effect:** Encourages **sparsity**. It can push the weights of unimportant features/leaves to exactly zero.
    *   **When to use:** Use when you have many features and want to perform automatic feature selection.
*   **`reg_lambda`** (L2 Regularization)
    *   **What it is:** Adds a penalty proportional to the **square** of the weights.
    *   **Effect:** Encourages **stability**. It shrinks all weights toward zero but rarely makes them exactly zero. It prevents any single leaf from having an extreme influence.
    *   **When to use:** Use to smooth out predictions and prevent the model from being too sensitive to noise.

**Summary Cheat Sheet**

| Parameter | Target | To Reduce Overfitting... |
| :--- | :--- | :--- |
| **`subsample`** | Rows | **Decrease** (e.g., 0.8) |
| **`subsample_freq`** | Rows | **Increase** (e.g., 1 or 5) |
| **`colsample_bytree`** | Columns | **Decrease** (e.g., 0.7) |
| **`min_child_samples`** | Leaves | **Increase** |
| **`reg_alpha`** | Weights | **Increase** |
| **`reg_lambda`** | Weights | **Increase** |

**`random_state=42,`**
Sets a seed for the random number generator. This ensures that the results are **reproducible**; every time you run the code with this seed, you will get the exact same model and results.

**`verbosity=-1`**
Controls the logging output. 
*   `1`: Displays info/warnings.
*   `0`: Displays only errors.
*   `-1`: **Silent mode** (suppresses all info, warnings, and messages, including the "No further splits" warnings you encountered).
#
---
---

## Learning Rate - Detail
In Gradient Boosting, the model builds trees **sequentially**. Each new tree is trained to predict the **errors** (residuals) made by the previous trees.

The `learning_rate` acts as a **scaling factor** that determines how much weight is given to the new tree's prediction when updating the overall model.

### The Mathematical Intuition
The final prediction of the model is the sum of the predictions from all individual trees:

$$\text{Final Prediction} = \text{Tree}_1 + (\eta \times \text{Tree}_2) + (\eta \times \text{Tree}_3) + \dots + (\eta \times \text{Tree}_n)$$

Where **$\eta$ (eta)** is your `learning_rate`.


### A Step-by-Step Example
Imagine you are trying to predict a target value of **10**.

1.  **Initial State:** The model starts with a baseline prediction of **5**.
    *   *Error (Residual):* $10 - 5 = \mathbf{5}$
2.  **Tree 1:** The first tree tries to predict that error of **5**. It predicts **4**.
    *   *Update:* $5 + (0.05 \times 4) = \mathbf{5.2}$
    *   *New Error:* $10 - 5.2 = \mathbf{4.8}$
3.  **Tree 2:** The second tree tries to predict the new error of **4.8**. It predicts **4.5**.
    *   *Update:* $5.2 + (0.05 \times 4.5) = \mathbf{5.425}$
    *   *New Error:* $10 - 5.425 = \mathbf{4.575}$

### The Trade-off: Why not just use 1.0?

| Learning Rate | Effect | Analogy |
| :--- | :--- | :--- |
| **High (e.g., 0.5 or 1.0)** | The model makes massive jumps toward the target. It learns very fast but often "overshoots" the optimal point, leading to **overfitting**. | Taking giant leaps toward a destination; you might jump right over it. |
| **Low (e.g., 0.01 or 0.05)** | The model makes tiny, careful adjustments. It is much more likely to find the exact "sweet spot" (minimum error) without overshooting. | Taking small, precise steps; you are much more likely to land exactly on the target. |

### The Golden Rule
There is an inverse relationship between `learning_rate` and `n_estimators`:
*   If you **decrease** the `learning_rate` (to make the model more precise), you **must increase** the `n_estimators` (to give the model enough steps to reach the target).
*   If you use a very low learning rate with a very low number of estimators, the model will "underfit" because it hasn't had enough steps to reach the correct prediction.
#
---
---